In [1]:
import pandas as pd
import sklearn
import numpy as np

### Import Data

In [3]:
df_jan = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')
df_mar = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet')

### Q1

In [4]:
len(df_jan.columns)

19

### Q2

In [5]:
df_jan['duration'] = (df_jan['tpep_dropoff_datetime'] - df_jan['tpep_pickup_datetime'] ) / pd.Timedelta(minutes=1)

In [6]:
df_jan['duration'].std()

42.59435124195458

### Q3

In [7]:
pre_outlier_row_total = df_jan.shape[0]

In [8]:
df_jan_no_outliers = df_jan.query('duration >= 1 and duration <= 60').reset_index(drop=True).copy()

In [9]:
post_outlier_row_total = df_jan_no_outliers.shape[0]
post_outlier_row_total / pre_outlier_row_total

0.9812202822125979

### Q4

In [10]:
from sklearn.preprocessing import OneHotEncoder

In [11]:
enc = OneHotEncoder(drop='first', handle_unknown='ignore')

In [12]:
enc_jan_trips = enc.fit_transform(df_jan_no_outliers[['PULocationID', 'DOLocationID']])

In [13]:
enc_jan_trips

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6011507 stored elements and shape (3009173, 513)>

### Q5

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [15]:
lin_reg = LinearRegression()

In [16]:
lin_reg.fit(enc_jan_trips, df_jan_no_outliers['duration'].values)

LinearRegression()

In [17]:
y_pred = lin_reg.predict(enc_jan_trips)

In [18]:
print(root_mean_squared_error(df_jan_no_outliers['duration'].values, y_pred))

7.6492617930636735


### Q6

In [22]:
df_mar['duration'] = (df_mar['tpep_dropoff_datetime'] - df_mar['tpep_pickup_datetime'] ) / pd.Timedelta(minutes=1)

In [23]:
df_mar_no_outliers = df_mar.query('duration >= 1 and duration <= 60').reset_index(drop=True).copy()

In [24]:
enc_mar_trips = enc.transform(df_mar_no_outliers[['PULocationID', 'DOLocationID']])

c:\Users\khanm375\Documents\mlops\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:241: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [25]:
y_val_pred = lin_reg.predict(enc_mar_trips)

In [26]:
print(root_mean_squared_error(df_mar_no_outliers['duration'].values, y_val_pred))

8.249553477796988


In [28]:
# Calculate the standard deviation of the predictions
y_val_pred.std()

6.247476842084266

### Question 2

In [29]:
df_result = pd.DataFrame()

year = 2023
month = 3

df_result['predictions'] = y_val_pred 
df_result['ride_id'] = f'{year:04d}/{month:02d}_' + df_mar_no_outliers.index.astype(str)

df_result.head()

,predictions,ride_id
0,16.245860,2023/03_0
1,26.135122,2023/03_1
2,11.884481,2023/03_2
3,11.997607,2023/03_3
4,10.234371,2023/03_4


In [30]:
df_result.to_parquet(
    "yellow_2023-03.parquet",
    engine='pyarrow',
    compression=None,
    index=False
)

In [33]:
from pathlib import Path 

Path("yellow_2023-03.parquet").stat().st_size  / 1024 / 1024  # size in MB

65.43695449829102